### Task 3 - Agentic Financial Research Workflow

#### System Architecture

```text
User Ticker Input
        ↓
Financial Tools Layer
- get_price_data()
- calculate_volatility()
- get_news()
- llm_sentiment()
- web_search()
        ↓
Quantitative Analyst Agent
        ↓
Sentiment Research Agent
        ↓
Risk Review / Critique Agent
        ↓
Portfolio Strategist Agent
        ↓
Structured Final Report
        ↓
Persistent Memory Cache
        ↓
Follow-Up Question Handling
```
#### Design Philosophy

The workflow intentionally uses explicit orchestration instead of autonomous planning loops.

This improves:
- explainability
- debugging
- observability
- structured validation
- reliability in notebook-based environments

Each agent has a focused responsibility and communicates through validated structured outputs.

In [10]:
import sys
from pathlib import Path
import importlib
import json

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "task3_agentic" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import task3_agentic.src.tracing
import task3_agentic.src.tools

# Jupyter notebooks cache imported modules aggressively.
# Reloading ensures the latest decorator/logging changes are used.
importlib.reload(task3_agentic.src.tracing)
importlib.reload(task3_agentic.src.tools)

from src.tools import get_price_data, calculate_volatility, get_news, llm_sentiment, web_search
from src.workflow import run_workflow, answer_follow_up_question

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# Testing Financial Data Tool
price_data = get_price_data("NVDA")

print(price_data["summary"])

{'ticker': 'NVDA', 'current_price': 219.44, 'sma_50': 189.49, 'sma_200': 184.95, 'rsi_14': 64.92, 'macd': 6.57, 'macd_signal': 5.84, 'bollinger_upper': 219.29, 'bollinger_lower': 190.08, 'momentum_signal': 'Bullish'}


In [7]:
# Testing Volatility Tool
volatility = calculate_volatility("NVDA")

print(volatility)

{'ticker': 'NVDA', 'annualized_volatility': 0.3633}


In [9]:
# Testing News Retrieval
news = get_news("NVDA")

print(news[:3])

[{'title': "Intel stock rises as CEO Lip-Bu Tan touts 'exciting new products' with Nvidia", 'source': 'Yahoo Finance', 'published': '2026-05-11T15:47:50Z'}, {'title': 'Memory chip stocks hit record highs, pharma reacts to hantavirus concerns', 'source': 'Yahoo Finance Video', 'published': '2026-05-11T14:54:55Z'}, {'title': 'Dow Jones Futures: AI, Chip Stocks Shrug Off Oil Prices, Trump Comments; CPI Inflation Due', 'source': "Investor's Business Daily", 'published': '2026-05-11T22:52:27Z'}]


In [11]:
# Testing LLM Sentiment Analysis
sentiment = llm_sentiment(news[:5])

print(sentiment)

{'results': [{'headline': "Intel stock rises as CEO Lip-Bu Tan touts 'exciting new products' with Nvidia", 'sentiment': 'positive', 'confidence': 0.9, 'reason': "CEO mentions 'exciting new products'"}, {'headline': 'Memory chip stocks hit record highs, pharma reacts to hantavirus concerns', 'sentiment': 'positive', 'confidence': 0.8, 'reason': 'Memory chip stocks hit record highs'}, {'headline': 'Dow Jones Futures: AI, Chip Stocks Shrug Off Oil Prices, Trump Comments; CPI Inflation Due', 'sentiment': 'neutral', 'confidence': 0.7, 'reason': 'No clear positive or negative sentiment'}, {'headline': "Should You Really Ignore Palantir's Steep Valuation and Buy the Stock? Here's What History Says.", 'sentiment': 'neutral', 'confidence': 0.6, 'reason': 'Questioning whether to buy the stock, no clear sentiment'}, {'headline': 'SoundHound AI Stock Falls After Earnings: Should You Buy the Dip?', 'sentiment': 'negative', 'confidence': 0.9, 'reason': 'Stock falls after earnings'}], 'overall_sentim

In [12]:
results = web_search("NVDA analyst concerns")

print(results[:2])

[{'title': 'NVDA Forecast — Price Target — Prediction for 2027 — TradingView', 'snippet': 'The 58 analysts offering 1-year price forecasts have a max estimate of — and a min estimate of —. Analyst rating. Based on 70 analysts giving stock ratings in the past 3 months.', 'url': 'https://www.tradingview.com/symbols/NASDAQ-NVDA/forecast/'}, {'title': 'NVIDIA Analyst Ratings and Price Targets | NASDAQ:NVDA | Benzinga', 'snippet': 'The analyst firm set a price target for $250.00 expecting NVDA to rise to within 12 months (a possible 27.10% upside). 100 analyst firms have reported ratings in the last year.', 'url': 'https://www.benzinga.com/quote/NVDA/analyst-ratings'}]


In [13]:
get_price_data("INVALID123")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: INVALID123"}}}
$INVALID123: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


ValueError: No data found for ticker: INVALID123

In [14]:
result = run_workflow(
    "NVDA",
    debug=False,
    refresh_cache=True
)

[workflow] Starting analysis for NVDA
[workflow] get_price_data...
[workflow] calculate_volatility...
[workflow] get_news...
[workflow] llm_sentiment...
[workflow] web_search...
[workflow] Running quantitative analyst agent...
[workflow] Running sentiment research agent...
[workflow] Running critique loop...
[workflow] Running portfolio strategist agent...
[workflow] Complete. Cache: C:\Users\poshi\Documents\My\Interview-quiz\cdazzdev-mla\task3_agentic\memory\NVDA_report_cache.json | Output: C:\Users\poshi\Documents\My\Interview-quiz\cdazzdev-mla\task3_agentic\outputs\NVDA_report_20260511_230211.json


In [15]:
print(json.dumps(
    result["final_report"],
    indent=2
))

{
  "ticker": "NVDA",
  "financial_health_summary": "NVDA has a positive sentiment and a bullish momentum signal, but high RSI value and current price position above the upper Bollinger Band suggest potential risks. The annualized volatility is relatively high, indicating a higher risk of price fluctuations.",
  "top_three_risks": [
    "High RSI value (64.92) may indicate overbought conditions.",
    "Current price is above the upper Bollinger Band, which may lead to a price correction.",
    "Concentration risk in Nvidia's client profile"
  ],
  "hedge_strategy": "Consider a 20-30% allocation to a diversified portfolio of large-cap stocks with lower volatility, such as Johnson & Johnson (JNJ) or Procter & Gamble (PG), to mitigate potential risks.",
  "evidence_used": [
    "Quantitative analysis",
    "Sentiment analysis",
    "Risk review"
  ],
  "confidence": 0.65
}


In [16]:
result2 = run_workflow("NVDA")

[workflow] Starting analysis for NVDA
[workflow] Cache hit for NVDA; skipping tools and agents.


In [17]:
answer_follow_up_question(
    "NVDA",
    "What are the main risks?"
)

[workflow] Answered follow-up for NVDA from cache only.


{'ticker': 'NVDA',
 'question': 'What are the main risks?',
 'answer': "The top cached risks are: High RSI value (64.92) may indicate overbought conditions.; Current price is above the upper Bollinger Band, which may lead to a price correction.; Concentration risk in Nvidia's client profile",
 'used_memory': True,
 'source': 'C:\\Users\\poshi\\Documents\\My\\Interview-quiz\\cdazzdev-mla\\task3_agentic\\memory\\NVDA_report_cache.json'}

In [18]:
answer_follow_up_question(
    "NVDA",
    "What is the hedge strategy?"
)

[workflow] Answered follow-up for NVDA from cache only.


{'ticker': 'NVDA',
 'question': 'What is the hedge strategy?',
 'answer': 'Consider a 20-30% allocation to a diversified portfolio of large-cap stocks with lower volatility, such as Johnson & Johnson (JNJ) or Procter & Gamble (PG), to mitigate potential risks.',
 'used_memory': True,
 'source': 'C:\\Users\\poshi\\Documents\\My\\Interview-quiz\\cdazzdev-mla\\task3_agentic\\memory\\NVDA_report_cache.json'}

## Observability and Trace Logging

In [19]:
trace_path = Path("logs/agent_trace.jsonl")

with open(trace_path, "r") as f:
    lines = f.readlines()

print(json.dumps(
    lines[-5:],
    indent=2
))

[
  "{\"timestamp\": \"2026-05-11T23:02:11.436063\", \"tool\": \"PortfolioStrategistAgent\", \"inputs\": {\"model\": \"llama-3.1-8b-instant\", \"task_prompt_preview\": \"Create the final investment outlook for ticker: NVDA\\n\\nQuantitative analysis:\\n{\\n  \\\"confidence\\\": 0.8,\\n  \\\"key_metrics\\\": {\\n    \\\"bollinger_bands\\\": {\\n      \\\"current_price\\\": 219.44,\\n      \\\"lower\\\": 190.08,\\n      \\\"upper\\\": 219.29\\n    },\\n    \\\"macd\\\": {\\n      \\\"macd_line\\\": 6.57,\\n      \\\"signal_line\\\": 5.84\\n    },\\n    \\\"momentum\\\": {\\n      \\\"signal\\\": \\\"Bullish\\\"\\n    },\\n    \\\"rsi\\\": {\\n      \\\"value\\\": 64.92\\n    },\\n    \\\"sma\\\": {\\n      \\\"200\\\": 184.95,\\n      \\\"50\\\": 189.49\\n    }\\n  },\\n  \\\"momentum_signal\\\": \\\"Bullish\\\",\\n  \\\"quantitati\"}, \"output_preview\": \"{'result': {'ticker': 'NVDA', 'financial_health_summary': 'NVDA has a positive sentiment and a bullish momentum signal, but high RSI